# Training_File_Floor.ipynb — STEP 2D Floor Diagnostic

**Branch git**: `Floor_Diagnostic` (eredita infra da `Optimizer_Exploration`).

**Contesto**: STEP 2C ha confermato che il plateau val~0.28 NON è risolvibile da:
- capacità (sweep STEP 2B su 11× hidden range)
- ottimizzatore (sweep STEP 2C: Prodigy lr=0.1, AdamW, comparable Δ=0.18%)
- scheduler (OneCycle vs none)

Dato che **8 setup diversi convergono nella banda 0.279-0.282**, il floor è **strutturale**.
Questo notebook identifica QUALE delle 4 cause candidate è responsabile.

**Configurazione baseline (vincente STEP 2C)**: AdamW b=8 + OneCycle + h=32, r=8 + 15 ep × 190 cap.

**Plan in ordine di costo/utilità** (Po2 DIFFERITA per scelta utente):

| Plan | Test | Modifica vs baseline | Costo Azure | Predict colpevole |
|------|------|----------------------|-------------|-------------------|
| **F1** | PINN no-multi-obj | `lambda_phys=lambda_ou=lambda_bc=0` | ~45 min | ~50% (most likely) |
| **F2** | OU noise OFF | `--noise_scale 0.0` (cache diverso) | ~45 min | ~15% |
| **F3** | Dataset 3.3× | `--n_train 5000` (cache nuovo, 3× più grande) | ~2-3h | ~10% |
| ~~F4~~ | ~~Po2 quantization OFF~~ | _differito (esperimenti utente prima)_ | — | ~25% |

**Output atteso**:
- Se F1 batte 0.27 → floor era PINN trade-off → STEP 2E: ribilanciare λ
- Se F2 batte 0.27 → floor era OU noise (irriducibile in produzione, ma sappiamo)
- Se F3 batte 0.27 → floor era data diversity → STEP 2E: dataset più grande/diverso
- Se TUTTI restano a 0.28 → floor è Po2 quantization (test F4 differito)

**Reference**: AdamW b=8 del branch Optimizer_Exploration ha val=0.2805. Lo useremo come baseline.

In [ ]:
# ===========================================================
# CELLA 0 — Bootstrap deps + git checkout Floor_Diagnostic
# ===========================================================
import sys

print('📦 Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas
!{sys.executable} -m pip install --quiet nbstripout

# nbstripout per evitare conflitti git sui notebook (output stripping)
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   ✅ deps installed + nbstripout attivato')

# Sync con remote
print('\n🔄 git fetch + checkout Floor_Diagnostic + pull:')
!git fetch origin
!git checkout Floor_Diagnostic 2>/dev/null || git checkout -b Floor_Diagnostic origin/Floor_Diagnostic
!git pull --no-rebase --no-edit origin Floor_Diagnostic

print('\n📍 Branch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 — ENV CHECK
# ===========================================================
import sys, platform, os

print(f'🐍 Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'📂 Working dir:     {os.getcwd()}')

checks = []
for mod in ['torch', 'numpy', 'pandas', 'matplotlib']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'   ✅ {mod:<14} v{v}')
        checks.append((mod, True))
    except Exception as e:
        print(f'   ❌ {mod:<14} {e}')
        checks.append((mod, False))

import torch
if torch.cuda.is_available():
    print(f'\n🎮 CUDA:            ✅ {torch.cuda.get_device_name(0)}')
else:
    print(f'\n🎮 CUDA:            ❌ training su CPU')

print('\n📁 File critici:')
for f in ['train.py', 'config.py', 'core/network.py', 'data/generator.py']:
    exists = os.path.isfile(f)
    print(f'   {"✅" if exists else "❌"} {f}')
    checks.append((f, exists))

# Verifica nuovo CLI --noise_scale
import subprocess
res = subprocess.run([sys.executable, 'train.py', '--help'], capture_output=True, text=True)
has_noise = 'noise_scale' in (res.stdout + res.stderr)
print(f'\n🔧 train.py supporta --noise_scale: {"✅" if has_noise else "❌ (branch sbagliato? rifare Cella 0)"}')
checks.append(('noise_scale CLI', has_noise))

n_fail = sum(1 for _, ok in checks if not ok)
if n_fail == 0:
    print('\n✅ ENV CHECK PASSED — procedi con Cella 2')
else:
    print(f'\n❌ ENV CHECK FAILED — {n_fail} problemi')
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 — FLOOR_PLAN: 3 config diagnostiche
# ===========================================================
# Configurazione comune: AdamW vincente di STEP 2C (Plan B Optimizer_Exploration).
# Ogni Plan varia UNA SOLA cosa per isolare il colpevole del floor.
#
# ⚠️  ATTENZIONE asimmetria F3: con n_train=5000 e stesso budget step (2850),
#     reps scende a ~0.10 (vs 0.32 di F1/F2). Se F3 non migliora val ma
#     SOSPETTI sia per under-exposure, puoi bumpare max_steps_per_epoch a 400
#     (= 6000 step → reps ~0.20, tempo 3h) — vedi nota sotto.

import time, subprocess, sys, os, datetime, shutil, glob, json
import pandas as pd, numpy as np

FLOOR_COMMON = {
    'n_train':              1500,       # F1, F2; F3 lo overrida
    'n_val':                300,
    'seq_len':              50,
    'scenario_mix':         'highway',
    'cut_in_ratio':         0.0,
    'cf_hidden_size':       32,
    'cf_rank':              8,
    # PINN weights nominali (F1 li overrida a 0)
    'lambda_data':          1.0,
    'lambda_phys':          0.1,
    'lambda_ou':            0.05,
    'lambda_bc':            1.0,
    # OU noise nominale (F2 lo overrida a 0)
    'noise_scale':          1.0,
    # AdamW vincente STEP 2C
    'optimizer':            'adamw',
    'lr':                   2e-3,
    'batch_size':           8,
    'val_batch_size':       64,
    'scheduler':            'onecycle',
    'max_lr':               2e-3,
    'epochs':               15,
    'max_steps_per_epoch':  190,        # 190 × 15 = 2850 step (= STEP 2C Plan B)
    # Safety: no abort, no early stop (vediamo tutto fino in fondo)
    'max_inf_streak':       99999,
    'early_stop_patience':  0,
    'early_stop_delta':     0.005,
}

FLOOR_PLAN = [
    # ── F1: PINN no-multi-obj (solo data loss) ──────────────────────
    # Ipotesi: il trade-off 5-obiettivi limita val. Disattivando phys/ou/bc
    # vediamo se val_data scende sotto 0.27.
    {
        'tag':            'P12_S2D_F1_no_pinn',
        'lambda_phys':    0.0,
        'lambda_ou':      0.0,
        'lambda_bc':      0.0,
        # lambda_data resta 1.0
    },
    # ── F2: OU noise OFF (dataset deterministico) ───────────────────
    # Ipotesi: l'OU noise (NOISE_GAP_REL/VEL_OPT/ACCEL) introduce errore
    # irriducibile. Con noise_scale=0 vediamo il floor 'puro' senza rumore.
    # Cache filename diverso (cache_1500_highway_cut0.0_ou0.0.pt) → fresh gen.
    {
        'tag':            'P12_S2D_F2_no_ou',
        'noise_scale':    0.0,
    },
    # ── F3: Dataset 3.3× (n_train=5000) ─────────────────────────────
    # Ipotesi: 1500 traiettorie non bastano. Con 5000 dataset 3.3× più grande.
    # NOTA: con stesso budget step (2850), reps cala a ~0.10 (vs 0.32 di F1/F2).
    # Se vuoi reps simili (~0.30), aggiungi 'max_steps_per_epoch': 600
    # (= 9000 step, ~3h Azure). Lasciato 190 per ora per non sforare il tempo.
    {
        'tag':            'P12_S2D_F3_dataset_big',
        'n_train':        5000,
    },
]

# Quali eseguire? Modifica per disabilitare singoli plan.
RUN_FLOOR = ['F1', 'F2', 'F3']

def _cache_path(plan):
    """Costruisce path cache. Include _ou<scale> se diverso da 1.0 per evitare
    collisione con cache esistenti generate con noise nominale."""
    full = {**FLOOR_COMMON, **plan}
    base = f'data/cache_{full["n_train"]}_{full["scenario_mix"]}_cut{full["cut_in_ratio"]}'
    if full['noise_scale'] != 1.0:
        base += f'_ou{full["noise_scale"]}'
    return f'{base}.pt'

print(f'🔬 FLOOR DIAGNOSTIC: {len(FLOOR_PLAN)} config')
print(f'⚠️  Po2 quantization NON testata (differita per scelta utente)')
print(f'⚠️  Safety: max_inf_streak=99999, early_stop=0')
print(f'📊 Reference: AdamW b=8 STEP 2C Plan B → val=0.2805 (per confronto)')

for i, plan in enumerate(FLOOR_PLAN, 1):
    label = f'F{i}'
    active = '✅' if label in RUN_FLOOR else '⏸️'
    full = {**FLOOR_COMMON, **plan}
    cp = _cache_path(plan)
    cache_exists = '📦 esistente' if os.path.isfile(cp) else '🆕 da generare'
    n_w_est = full['n_train'] * 47  # ~47 windows/traj
    n_steps = full['max_steps_per_epoch'] * full['epochs']
    reps    = n_steps * full['batch_size'] / n_w_est
    reps_warn = ' ⚠️ low reps!' if reps < 0.15 else ''
    print(f'\n  {active} {label}: {full["tag"]}')
    print(f'     override: {list(plan.keys())[1:] if len(plan) > 1 else "(see lambda)"}')
    print(f'     lambda_data={full["lambda_data"]} phys={full["lambda_phys"]} ou={full["lambda_ou"]} bc={full["lambda_bc"]}')
    print(f'     n_train={full["n_train"]}  noise_scale={full["noise_scale"]}')
    print(f'     cache: {cp}  ({cache_exists})')
    print(f'     budget: {full["max_steps_per_epoch"]}/ep × {full["epochs"]} ep = {n_steps} step  (reps≈{reps:.3f}{reps_warn})')

In [ ]:
# ===========================================================
# CELLA 3 — RUN sweep diagnostic (continue-on-failure)
# ===========================================================
# ⚠️  SKIP_IF_EXISTS=True (default): se results/<tag>/training_log.csv esiste
#     gia', il plan viene SALTATO. Permette di rieseguire la cella dopo aver
#     interrotto/aggiunto un plan senza ripetere quelli completati. Per forzare
#     la ri-esecuzione, set SKIP_IF_EXISTS=False (o cancella results/<tag>/).
SKIP_IF_EXISTS = True

def _build_cli_floor(plan):
    full = {**FLOOR_COMMON, **plan}
    args = [
        sys.executable, 'train.py',
        '--epochs',              str(full['epochs']),
        '--n_train',             str(full['n_train']),
        '--n_val',               str(full['n_val']),
        '--batch_size',          str(full['batch_size']),
        '--val_batch_size',      str(full['val_batch_size']),
        '--seq_len',             str(full['seq_len']),
        '--scheduler',           full['scheduler'],
        '--max_lr',              str(full['max_lr']),
        '--lr',                  str(full['lr']),
        '--optimizer',           full['optimizer'],
        '--scenario_mix',        full['scenario_mix'],
        '--cut_in_ratio',        str(full['cut_in_ratio']),
        '--cf_hidden_size',      str(full['cf_hidden_size']),
        '--cf_rank',             str(full['cf_rank']),
        '--lambda_data',         str(full['lambda_data']),
        '--lambda_phys',         str(full['lambda_phys']),
        '--lambda_ou',           str(full['lambda_ou']),
        '--lambda_bc',           str(full['lambda_bc']),
        '--noise_scale',         str(full['noise_scale']),
        '--max_inf_streak',      str(full['max_inf_streak']),
        '--early_stop_patience', str(full['early_stop_patience']),
        '--early_stop_delta',    str(full['early_stop_delta']),
        '--max_steps_per_epoch', str(full['max_steps_per_epoch']),
        '--data_cache',          _cache_path(plan),
        '--tag',                 full['tag'],
    ]
    return args

def _push_floor_result(plan):
    full = {**FLOOR_COMMON, **plan}
    tag = full['tag']
    src, dst = f'checkpoints/{tag}', f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   ⚠️  {src} mancante')
        return False
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str, status = '', 'unknown'
    if os.path.isfile(f'{dst}/training_log.csv'):
        edf = pd.read_csv(f'{dst}/training_log.csv')
        if len(edf) > 0:
            val_str = f'best val={edf.val_total.min():.4f} (E{int(edf.val_total.idxmin())+1}/{len(edf)})'
            status = 'FROZEN' if (len(edf) > 2 and edf.val_total.iloc[1:].nunique() == 1) else 'OK'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (S2D Floor): {tag} ({ts})\n\n'
        f'Status: {status}  |  {val_str}\n'
        f'Override: lambda_phys={full["lambda_phys"]} ou={full["lambda_ou"]} bc={full["lambda_bc"]}, '
        f'noise_scale={full["noise_scale"]}, n_train={full["n_train"]}\n'
        f'Cache: {_cache_path(plan)}\n'
        f'Branch Floor_Diagnostic\n'
    )
    with open('/tmp/floor_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst}
    !git commit -F /tmp/floor_msg.txt
    !rm /tmp/floor_msg.txt
    !git pull --no-rebase --no-edit origin Floor_Diagnostic
    !git push origin Floor_Diagnostic
    return True

# ── Esecuzione ─────────────────────────────────────────────────
floor_results = []
t_start = time.time()

for i, plan in enumerate(FLOOR_PLAN, 1):
    label = f'F{i}'
    if label not in RUN_FLOOR:
        print(f'\n⏸️  Skipping {label} ({plan["tag"]}) — non in RUN_FLOOR')
        continue
    full = {**FLOOR_COMMON, **plan}
    tag  = full['tag']
    # SKIP_IF_EXISTS: salta se i risultati sono gia' sotto results/<tag>/
    if SKIP_IF_EXISTS and os.path.isfile(f'results/{tag}/training_log.csv'):
        try:
            ep_existing = pd.read_csv(f'results/{tag}/training_log.csv')
            v_existing = ep_existing.val_total.min() if len(ep_existing) else None
            v_str = f'val={v_existing:.4f}' if v_existing is not None else 'empty'
        except Exception:
            v_str = 'unreadable'
        print(f'\n⏭️  SKIP {label}: results/{tag}/ gia\' presente ({v_str}). '
              f'Set SKIP_IF_EXISTS=False per ri-eseguire.')
        floor_results.append({'label': label, 'tag': tag, 'status': 'skipped_existing', 'elapsed': 0.0})
        continue
    print('\n' + '=' * 78)
    print(f'🔬 {label}: {tag}')
    print('=' * 78)
    cmd = _build_cli_floor(plan)
    print(f'CLI: {" ".join(cmd[2:])}\n')
    t0 = time.time()
    res = subprocess.run(cmd, capture_output=False)
    elapsed = time.time() - t0
    status = 'ok' if res.returncode == 0 else f'fail (exit {res.returncode})'
    print(f'\n⏱️  {label} terminato in {elapsed/60:.1f}min — exit: {status}')
    print(f'\n📤 Push results/{tag}...')
    _push_floor_result(plan)
    floor_results.append({'label': label, 'tag': tag, 'status': status, 'elapsed': elapsed})

print(f'\n{"="*78}\n🏁 FLOOR SWEEP COMPLETATO in {(time.time()-t_start)/60:.1f} min\n{"="*78}')
for r in floor_results:
    print(f"   {r['label']:<4} {r['tag']:<32} {r['status']:<20} {r['elapsed']/60:.1f}min")

In [ ]:
# ===========================================================
# CELLA 4 — ANALISI: confronto F1/F2/F3 vs baseline AdamW
# ===========================================================
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

BASELINE_TAG = 'P12_S2C_planB_adamw_b8'   # da branch Optimizer_Exploration
BASELINE_VAL = 0.2805

# Carica risultati floor
loaded = []
for plan in FLOOR_PLAN:
    full = {**FLOOR_COMMON, **plan}
    tag = full['tag']
    base = f'results/{tag}'
    if not os.path.isdir(base):
        print(f'⏭️  {tag}: no results, skip')
        continue
    ep_csv = f'{base}/training_log.csv'
    if not os.path.isfile(ep_csv):
        continue
    ep = pd.read_csv(ep_csv)
    loaded.append({'plan': plan, 'full': full, 'ep': ep})

# Carica baseline AdamW (se presente)
baseline_ep = None
if os.path.isdir(f'results/{BASELINE_TAG}'):
    baseline_ep = pd.read_csv(f'results/{BASELINE_TAG}/training_log.csv')

if not loaded:
    print('❌ Nessun risultato Floor. Esegui Cella 3 prima.')
else:
    out_dir = f'opt_plots/floor_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}'
    os.makedirs(out_dir, exist_ok=True)

    # ── Tabella riassuntiva ────────────────────────────────────
    rows = []
    for d in loaded:
        full, ep = d['full'], d['ep']
        best = ep.val_total.min()
        delta = best - BASELINE_VAL
        verdict = '🎯 COLPEVOLE!' if delta < -0.01 else ('≈ comparable' if abs(delta) < 0.005 else '❌ peggio')
        rows.append({
            'tag':       full['tag'].replace('P12_S2D_', ''),
            'override':  ('PINN=0' if full['lambda_phys']==0 else
                          'OU=0' if full['noise_scale']==0 else
                          f'n_train={full["n_train"]}'),
            'best_val':  f'{best:.4f}',
            'vs_baseline': f'{delta:+.4f}',
            'verdict':   verdict,
        })
    df = pd.DataFrame(rows)
    display(Markdown(f'### 📊 Floor Diagnostic — baseline AdamW val={BASELINE_VAL:.4f}'))
    display(df)

    # ── Grafico O1: val_total per epoca (overlay tutti + baseline) ──
    fig, ax = plt.subplots(figsize=(11, 6))
    cmap = plt.cm.tab10(np.linspace(0, 1, len(loaded)))
    for i, d in enumerate(loaded):
        ep = d['ep']
        lbl = d['full']['tag'].replace('P12_S2D_', '')
        ax.plot(ep.epoch, ep.val_total, 'o-', linewidth=2, markersize=7,
                color=cmap[i], label=lbl)
    if baseline_ep is not None:
        ax.plot(baseline_ep.epoch, baseline_ep.val_total, 'k--', linewidth=2.5,
                label=f'baseline AdamW ({BASELINE_VAL:.4f})')
    ax.axhline(BASELINE_VAL, ls=':', color='red', alpha=0.5,
               label=f'floor 2C (0.2805)')
    ax.axhline(0.20, ls=':', color='green', alpha=0.5, label='target val<0.20')
    ax.set_xlabel('epoch'); ax.set_ylabel('val_total')
    ax.set_title('Floor Diagnostic — val_total per epoca')
    ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'{out_dir}/F_val_per_epoch.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── Verdetto ───────────────────────────────────────────────
    display(Markdown('### 🎯 Verdetto'))
    best = min(loaded, key=lambda d: d['ep'].val_total.min())
    best_v = best['ep'].val_total.min()
    delta = best_v - BASELINE_VAL
    if delta < -0.05:
        print(f'🏆 FLOOR SUPERATO: {best["full"]["tag"]} con val={best_v:.4f} ({delta:+.4f})')
        print(f'   Causa floor identificata. Procedere con STEP 2E (mitigation).')
    elif delta < -0.01:
        print(f'✅ MIGLIORAMENTO PARZIALE: {best["full"]["tag"]} val={best_v:.4f} ({delta:+.4f})')
        print(f'   Floor parzialmente spiegato. Potrebbero esserci più cause sovrapposte.')
    else:
        print(f'❌ TUTTI E 3 RESTANO AL FLOOR (best={best_v:.4f}, Δ={delta:+.4f})')
        print(f'   Floor NON è PINN/OU/data. → Po2 quantization (F4 differita).')
    print(f'\n💾 Grafici: {out_dir}/')

# 🔬 STEP 2D-bis — Decomposizione del residuo 0.226

**Contesto post-F2** (2026-05-31): F2 (no-OU) ha portato val da 0.2805 → 0.2262 (Δ=-0.054).
Quindi **OU spiega 21% del floor**. Resta un residuo di **~0.226** ancora ignoto.

**Obiettivo Cella 7-9**: capire la composizione del residuo. Test 3 combinazioni:

| # | Tag | OU | spike_reg | Po2 | Cosa misura |
|---|-----|----|-----------|----|---|
| F5 | no_ou + no_sr | OFF | OFF | ON | Solo spike regularizer (lambda_sr=0) |
| F6 | no_ou + no_po2 | OFF | ON | OFF | Solo quantizzazione Po2 |
| F7 | no_ou + no_sr + no_po2 | OFF | OFF | OFF | "Floor pulito" (data+phys+architettura) |

**Decomposizione attesa**:
```
floor totale 0.28 = OU(0.054) + spike_reg(?) + Po2(?) + architettura/capacity(?)
                    └─F2─┘     └F5 vs F2 ┘   └F6 vs F2┘  └────── F7 ─────────┘
```

**⚠️ NOTA Po2** (importante per l'utente): l'utente vuole capire il PESO di Po2 ma NON
rimuoverla in deploy. Il toggle è 100% reversibile: `--po2_enabled 0/1` flag CLI legge
env var `PO2_ENABLED` LIVE ad ogni forward call. Dopo i test, ogni run successivo
con flag default (`--po2_enabled 1` o assente) ritorna automaticamente al comportamento
legacy. **Non c'è alcuna modifica permanente al codice.**

**Tempi stimati Azure CPU**: ~45 min × 3 = ~2h totali (cache F2 già disponibile, no rigenerazione).

In [ ]:
# ===========================================================
# CELLA 7 — FLOOR_PLAN_RES: 3 config per decomporre il residuo
# ===========================================================
# Eredita FLOOR_COMMON da Cella 2 (se Cella 2 non è stata eseguita, falla prima).
# Override solo lambda_sr e po2_enabled. noise_scale=0 in tutti (eredita da F2).

FLOOR_PLAN_RES = [
    # ── F5: no_ou + no_sr ───────────────────────────────────────────
    # Quanto pesa il spike-rate regularizer (lambda_sr=0.5 default → 0.0)?
    # noise_scale=0 → riusa la cache F2 già esistente.
    {
        'tag':            'P12_S2D_F5_no_ou_no_sr',
        'noise_scale':    0.0,
        'lambda_sr':      0.0,    # NUOVO override
        'po2_enabled':    1,      # legacy (default)
    },
    # ── F6: no_ou + no_po2 ──────────────────────────────────────────
    # Quanto pesa la quantizzazione Po2 sui pesi? --po2_enabled 0 bypassa.
    {
        'tag':            'P12_S2D_F6_no_ou_no_po2',
        'noise_scale':    0.0,
        'lambda_sr':      0.5,    # default (B5)
        'po2_enabled':    0,      # BYPASS Po2
    },
    # ── F7: no_ou + no_sr + no_po2 (floor pulito) ───────────────────
    # "Cosa rimane se tolgo tutti i fattori non-essenziali"
    {
        'tag':            'P12_S2D_F7_no_ou_no_sr_no_po2',
        'noise_scale':    0.0,
        'lambda_sr':      0.0,
        'po2_enabled':    0,
    },
]

# Aggiungiamo lambda_sr a FLOOR_COMMON se non esiste (backward-compat per Cella 2)
if 'lambda_sr' not in FLOOR_COMMON:
    FLOOR_COMMON['lambda_sr'] = 0.5    # default da config.py (B5)
if 'po2_enabled' not in FLOOR_COMMON:
    FLOOR_COMMON['po2_enabled'] = 1    # legacy

RUN_FLOOR_RES = ['F5', 'F6', 'F7']     # tutti

print(f'🔬 FLOOR RESIDUAL DECOMPOSITION: {len(FLOOR_PLAN_RES)} config')
print(f'📊 Baseline F2 (no-OU): val=0.2262')
print(f'🎯 Obiettivo: capire la composizione del residuo 0.226')
print(f'⚠️  Po2 BYPASS è 100% reversibile (--po2_enabled 1 ripristina legacy)')

for i, plan in enumerate(FLOOR_PLAN_RES, 5):    # numerazione F5, F6, F7
    label = f'F{i}'
    active = '✅' if label in RUN_FLOOR_RES else '⏸️'
    full = {**FLOOR_COMMON, **plan}
    cp = _cache_path({'noise_scale': full['noise_scale'], 'n_train': full['n_train'],
                       'scenario_mix': full['scenario_mix'], 'cut_in_ratio': full['cut_in_ratio']})
    cache_exists = '📦 esistente (F2 cache)' if os.path.isfile(cp) else '🆕 da generare'
    n_steps = full['max_steps_per_epoch'] * full['epochs']
    print(f'\n  {active} {label}: {full["tag"]}')
    print(f'     lambda_data={full["lambda_data"]} phys={full["lambda_phys"]} ou={full["lambda_ou"]} bc={full["lambda_bc"]} '
          f'sr={full["lambda_sr"]}')
    print(f'     noise_scale={full["noise_scale"]}  po2_enabled={full["po2_enabled"]}')
    print(f'     cache: {cp}  ({cache_exists})')
    print(f'     budget: {n_steps} step totali')

In [ ]:
# ===========================================================
# CELLA 8 — RUN sweep residuo F5/F6/F7
# ===========================================================
# ⚠️  SKIP_IF_EXISTS_RES=True (default): salta i plan F5/F6/F7 i cui results/
#     sono gia' presenti. Permette di rilanciare la cella dopo crash/interruzioni
#     senza ripetere lavoro. Per forzare, set SKIP_IF_EXISTS_RES=False.
SKIP_IF_EXISTS_RES = True

def _build_cli_floor_res(plan):
    full = {**FLOOR_COMMON, **plan}
    args = [
        sys.executable, 'train.py',
        '--epochs',              str(full['epochs']),
        '--n_train',             str(full['n_train']),
        '--n_val',               str(full['n_val']),
        '--batch_size',          str(full['batch_size']),
        '--val_batch_size',      str(full['val_batch_size']),
        '--seq_len',             str(full['seq_len']),
        '--scheduler',           full['scheduler'],
        '--max_lr',              str(full['max_lr']),
        '--lr',                  str(full['lr']),
        '--optimizer',           full['optimizer'],
        '--scenario_mix',        full['scenario_mix'],
        '--cut_in_ratio',        str(full['cut_in_ratio']),
        '--cf_hidden_size',      str(full['cf_hidden_size']),
        '--cf_rank',             str(full['cf_rank']),
        '--lambda_data',         str(full['lambda_data']),
        '--lambda_phys',         str(full['lambda_phys']),
        '--lambda_ou',           str(full['lambda_ou']),
        '--lambda_bc',           str(full['lambda_bc']),
        '--lambda_sr',           str(full['lambda_sr']),       # NUOVO override
        '--noise_scale',         str(full['noise_scale']),
        '--po2_enabled',         str(full['po2_enabled']),     # NUOVO override
        '--max_inf_streak',      str(full['max_inf_streak']),
        '--early_stop_patience', str(full['early_stop_patience']),
        '--early_stop_delta',    str(full['early_stop_delta']),
        '--max_steps_per_epoch', str(full['max_steps_per_epoch']),
        '--data_cache',          _cache_path({'noise_scale': full['noise_scale'],
                                              'n_train': full['n_train'],
                                              'scenario_mix': full['scenario_mix'],
                                              'cut_in_ratio': full['cut_in_ratio']}),
        '--tag',                 full['tag'],
    ]
    return args

def _push_floor_res(plan):
    full = {**FLOOR_COMMON, **plan}
    tag = full['tag']
    src, dst = f'checkpoints/{tag}', f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   ⚠️  {src} mancante')
        return False
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str = ''
    if os.path.isfile(f'{dst}/training_log.csv'):
        edf = pd.read_csv(f'{dst}/training_log.csv')
        if len(edf) > 0:
            val_str = f'best val={edf.val_total.min():.4f} (E{int(edf.val_total.idxmin())+1}/{len(edf)})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (S2D-bis Floor Residual): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'Override: lambda_sr={full["lambda_sr"]}, po2_enabled={full["po2_enabled"]}, '
        f'noise_scale={full["noise_scale"]}\n'
        f'Branch Floor_Diagnostic\n'
    )
    with open('/tmp/floor_res_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst}
    !git commit -F /tmp/floor_res_msg.txt
    !rm /tmp/floor_res_msg.txt
    !git pull --no-rebase --no-edit origin Floor_Diagnostic
    !git push origin Floor_Diagnostic
    return True

# ── Esecuzione ─────────────────────────────────────────────────
floor_res_results = []
t_start = time.time()

for i, plan in enumerate(FLOOR_PLAN_RES, 5):
    label = f'F{i}'
    if label not in RUN_FLOOR_RES:
        print(f'\n⏸️  Skipping {label} ({plan["tag"]}) — non in RUN_FLOOR_RES')
        continue
    full = {**FLOOR_COMMON, **plan}
    tag  = full['tag']
    # SKIP_IF_EXISTS_RES: salta se results/<tag>/training_log.csv esiste
    if SKIP_IF_EXISTS_RES and os.path.isfile(f'results/{tag}/training_log.csv'):
        try:
            ep_existing = pd.read_csv(f'results/{tag}/training_log.csv')
            v_existing = ep_existing.val_total.min() if len(ep_existing) else None
            v_str = f'val={v_existing:.4f}' if v_existing is not None else 'empty'
        except Exception:
            v_str = 'unreadable'
        print(f'\n⏭️  SKIP {label}: results/{tag}/ gia\' presente ({v_str}). '
              f'Set SKIP_IF_EXISTS_RES=False per ri-eseguire.')
        floor_res_results.append({'label': label, 'tag': tag, 'status': 'skipped_existing', 'elapsed': 0.0})
        continue
    print('\n' + '=' * 78)
    print(f'🔬 {label}: {tag}')
    print(f'   lambda_sr={full["lambda_sr"]}  po2_enabled={full["po2_enabled"]}')
    print('=' * 78)
    cmd = _build_cli_floor_res(plan)
    print(f'CLI: {" ".join(cmd[2:])}\n')
    t0 = time.time()
    res = subprocess.run(cmd, capture_output=False)
    elapsed = time.time() - t0
    status = 'ok' if res.returncode == 0 else f'fail (exit {res.returncode})'
    print(f'\n⏱️  {label} terminato in {elapsed/60:.1f}min — exit: {status}')
    print(f'\n📤 Push results/{tag}...')
    _push_floor_res(plan)
    floor_res_results.append({'label': label, 'tag': tag, 'status': status, 'elapsed': elapsed})

print(f'\n{"=" * 78}\n🏁 FLOOR RES COMPLETATO in {(time.time()-t_start)/60:.1f} min\n{"=" * 78}')
for r in floor_res_results:
    print(f"   {r['label']:<4} {r['tag']:<40} {r['status']:<20} {r['elapsed']/60:.1f}min")

In [ ]:
# ===========================================================
# CELLA 9 — ANALISI DECOMPOSIZIONE RESIDUO
# Floor totale ≈ OU + spike_reg + Po2 + architettura (residuo finale)
# ===========================================================
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Carica TUTTI i floor (originali F1/F2/F3 + nuovi F5/F6/F7) + baseline
TAGS_ALL = {
    'REF':  ('P12_S2C_planB_adamw_b8',         'AdamW baseline (PINN+OU+Po2+SR all ON)'),
    'F1':   ('P12_S2D_F1_no_pinn',             'no PINN (data only)'),
    'F2':   ('P12_S2D_F2_no_ou',               'no OU'),
    'F3':   ('P12_S2D_F3_dataset_big',         'n_train=5000'),
    'F5':   ('P12_S2D_F5_no_ou_no_sr',         'no OU + no SR'),
    'F6':   ('P12_S2D_F6_no_ou_no_po2',        'no OU + no Po2'),
    'F7':   ('P12_S2D_F7_no_ou_no_sr_no_po2',  'no OU + no SR + no Po2 (floor pulito)'),
}

vals = {}
for lbl, (tag, _) in TAGS_ALL.items():
    p = f'results/{tag}/training_log.csv'
    if os.path.isfile(p):
        ep = pd.read_csv(p)
        vals[lbl] = ep.val_total.min() if len(ep) else None
    else:
        vals[lbl] = None

# ── Tabella riassuntiva ───────────────────────────────────────────
rows = []
ref = vals.get('REF')
f2  = vals.get('F2')
for lbl, (tag, desc) in TAGS_ALL.items():
    v = vals.get(lbl)
    if v is None:
        rows.append({'lbl': lbl, 'desc': desc, 'best_val': '–', 'Δ vs REF': '–', 'Δ vs F2': '–'})
    else:
        delta_ref = v - ref if ref else 0
        delta_f2  = v - f2 if f2 else 0
        rows.append({
            'lbl': lbl, 'desc': desc,
            'best_val': f'{v:.4f}',
            'Δ vs REF': f'{delta_ref:+.4f}',
            'Δ vs F2':  f'{delta_f2:+.4f}',
        })
df = pd.DataFrame(rows)
display(Markdown('### 📊 Tabella decomposizione completa'))
display(df)

# ── Decomposizione quantitativa del floor ─────────────────────────
display(Markdown('### 🧮 Decomposizione quantitativa'))
if all(vals.get(k) is not None for k in ['REF','F2','F5','F6','F7']):
    ou_weight     = ref - f2
    sr_weight_a   = f2 - vals['F5']     # quando solo SR cambia (F2→F5)
    po2_weight_a  = f2 - vals['F6']     # quando solo Po2 cambia (F2→F6)
    sr_po2_weight = f2 - vals['F7']     # entrambi (F2→F7)
    residuo_arch  = vals['F7']           # residuo finale = capacity/architettura/data
    interaction   = sr_po2_weight - sr_weight_a - po2_weight_a
    print(f'  OU noise:        {ou_weight:+.4f}  ({100*ou_weight/ref:5.1f}% del floor totale)')
    print(f'  Spike-rate reg:  {sr_weight_a:+.4f}  ({100*sr_weight_a/ref:5.1f}%)')
    print(f'  Po2 quantize:    {po2_weight_a:+.4f}  ({100*po2_weight_a/ref:5.1f}%)')
    print(f'  ── Interazione SR×Po2: {interaction:+.4f} (positiva = sinergica, negativa = compensativa)')
    print(f'  Residuo architettura (F7): {residuo_arch:.4f}  ({100*residuo_arch/ref:5.1f}% del floor — IRRIDUCIBILE in questo setup)')
else:
    print('  ⚠️  Alcuni run mancanti, decomposizione parziale')

# ── Plot stacked decomposizione ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Bar chart: best_val per config (ordinati per val crescente)
ax = axes[0]
items = [(lbl, vals[lbl], desc) for lbl, (tag, desc) in TAGS_ALL.items() if vals.get(lbl) is not None]
items_sorted = sorted(items, key=lambda x: x[1], reverse=True)
labels = [f"{lbl}\n{desc[:25]}" for lbl, _, desc in items_sorted]
values = [v for _, v, _ in items_sorted]
colors = ['gray' if l.startswith('REF') else 'tab:blue' for l, _, _ in items_sorted]
ax.barh(labels, values, color=colors)
ax.set_xlabel('best val_total (lower = better)')
ax.set_title('Confronto val per config')
ax.axvline(ref, ls='--', color='red', alpha=0.5, label=f'REF={ref:.4f}')
ax.axvline(f2, ls=':', color='green', alpha=0.5, label=f'F2={f2:.4f}')
ax.legend()
ax.grid(alpha=0.3, axis='x')

# (b) Stacked decomposition
ax = axes[1]
if all(vals.get(k) is not None for k in ['REF','F2','F5','F6','F7']):
    components = ['Residuo\n(arch+data)', 'Po2', 'Spike Reg', 'OU noise']
    weights = [residuo_arch, po2_weight_a, sr_weight_a, ou_weight]
    bottoms = np.cumsum([0] + weights[:-1])
    cmap_d = plt.cm.viridis(np.linspace(0.2, 0.8, len(components)))
    for c, w, b, col in zip(components, weights, bottoms, cmap_d):
        ax.barh(['Floor 0.28'], [w], left=[b], color=col, label=f'{c}: {w:.4f} ({100*w/ref:.1f}%)')
    ax.set_xlim(0, ref * 1.05)
    ax.set_xlabel('val_total')
    ax.set_title('Decomposizione del floor (REF=0.28)')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'opt_plots/floor_decomposition_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}.png',
            dpi=120, bbox_inches='tight')
plt.show()

# ── Verdetto ──────────────────────────────────────────────────────
display(Markdown('### 🎯 Verdetto operativo'))
if all(vals.get(k) is not None for k in ['REF','F2','F5','F6','F7']):
    if residuo_arch < 0.20:
        print(f'🏆 Architettura capace di val < 0.20 ({residuo_arch:.4f}) → si può andare oltre il floor.')
        print(f'   → STEP 2E: cercare strategie per "compensare" OU+SR+Po2 senza eliminarli.')
    elif residuo_arch < 0.225:
        print(f'✅ Residuo architettura {residuo_arch:.4f} < 0.225: rete migliorabile con tweaks.')
    else:
        print(f'⚠️  Residuo architettura {residuo_arch:.4f}: capacità SNN potrebbe essere il limite.')
        print(f'   → STEP 2E: testare architettura con più layer / migliori delays / pre-training.')
    print(f'\n   Po2 contributo: {100*po2_weight_a/ref:.1f}% del floor.')
    print(f'   L\'utente NON intende rimuovere Po2 in deploy. Po2_enabled=1 resta default.')